In [7]:
import yfinance as yf

from google.adk.agents import Agent
from google.adk.events import Event
from google.adk.runners import Runner
import google.adk as adk
from google.adk.tools import google_search
from google.adk.sessions import InMemorySessionService, Session
from google.genai.types import Content, Part


In [2]:
import os
PROJECT_ID = "adk-build-1"             # @param {type:"string"}
LOCATION = "us-central1"               # @param {type:"string"}

# Set environment variables for the ADK and gcloud
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "True"

print(f"\n✅ Vertex AI configured for project '{PROJECT_ID}' in '{LOCATION}'.")


✅ Vertex AI configured for project 'adk-build-1' in 'us-central1'.


In [3]:
def get_stock_data(stock_symbol: str):
    """Get the stock data for a given stock symbol"""
    return yf.Ticker(stock_symbol).info

In [4]:
stock_data_agent = Agent(
    name="stock_data_agent",
    model="gemini-2.5-flash",
    description="Fetch important data points and ratios of a given stock.",
    instruction="You are a stock research assistant. You are given a stock symbol and you need to return the company's pe ratio, market cap, pb ratio, ebidta.",
    tools=[get_stock_data]
)

In [8]:
async def run_agent(agent: Agent, query: str, session: Session, user_id: str, is_router: bool = False) -> str:
    """Initializes a runner and executes a query for an agent and sessiion"""

    runner = Runner(
        agent=agent,
        session_service=session_service,
        app_name=agent.name
    )

    final_response = ""

    try:
        async for event in runner.run_async(
            user_id=user_id,
            session_id=session.id,
            new_message=Content(parts=[Part(text=query)], role="user"),
        ):
            if not is_router:
                print(f"EVENT: {event}")
            if event.is_final_response():
                final_response = event.content.parts[0].text

    except Exception as e:
        final_response = f"Error has occurred: {e}"

    if not is_router:
        print("\n"+"_"*50)
        print("Final response")
        print(final_response)
        print("\n"+"_"*50)

    return final_response

session_service = InMemorySessionService()
my_user_id = "stock_client_001"


In [9]:
async def run_stock_data_agent():
    session = await session_service.create_session(app_name=stock_data_agent.name, user_id=my_user_id)
    stock="AAPL"
    await run_agent(stock_data_agent, stock, session, my_user_id)

await run_stock_data_agent()

EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      function_call=FunctionCall(
        args={
          'stock_symbol': 'AAPL'
        },
        id='adk-4708ce63-6bd2-459f-a9e2-88c95d6ed968',
        name='get_stock_data'
      ),
      thought_signature=b'\n\xfb\x02\x01\x8f=k_\x1ak\xd9\xc2zo\xa7\xd2\x1d\xf5\xca\xe5\xfd\xdf4[\xc0D\xfd[P\r\xc2\xc2\xa8y\xbb\xb8\x15=Lv\x0eFP\xbd[\xdc\xa70o\xe5\xbc\xb1\xf5\xaeH,\x89\xc3s\xfa\x15\x05W\x84\xf0\xccs\xc5\t\xdd\x88\xcc\xfa\x14\x984\xfb\xf3g\xff\xa9B\xd3\x82\xa9\x89F\xe1\xee\xeau\xa4i\x1c\xb5b\xd1...'
    ),
  ],
  role='model'
) grounding_metadata=None partial=None turn_complete=None finish_reason=<FinishReason.STOP: 'STOP'> error_code=None error_message=None interrupted=None custom_metadata=None usage_metadata=GenerateContentResponseUsageMetadata(
  candidates_token_count=9,
  candidates_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=9
    ),
  ],
  promp

In [10]:
web_search_agent = Agent(
    name="web_search_agent",
    model="gemini-2.5-flash",
    description="Find stock-related news and recent updates from the internet.",
    instruction="""
        role: Find stock-related news and recent updates from the internet.
        "Always include sources.",
        "Prefer official reports over general news.",
        "Summarize findings concisely."
    """,
    tools=[google_search]
)

In [12]:
async def run_web_search_agent():
    session = await session_service.create_session(app_name=web_search_agent.name, user_id=my_user_id)
    stock="AAPL"
    await run_agent(web_search_agent, stock, session, my_user_id)

await run_web_search_agent()

EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""Apple Inc. (AAPL) is currently trading at approximately $254.11 as of March 25, 2026, marking a 1% increase for the day. The company's market capitalization stands at $3.69 trillion, with its next earnings report scheduled for April 30, 2026, for Q2 FY2026.

**Analyst Sentiment and Price Targets:**
Wall Street analysts maintain a "Buy" consensus rating for AAPL, with an average 12-month price target of around $295.31 to $304.40, suggesting a potential upside of approximately 16.9% to 19.93% from current levels. Individual price targets range from a low of $200 to a high of $350. This consensus is based on ratings from 24 to 42 analysts, with a significant portion recommending "Strong Buy" or "Buy".

**Recent Financial Performance:**
For Q1 FY2026 (ending December 27, 2025), Apple reported total revenue of $143.8 billion, a 16% year-over-year growth, and diluted earnings per share of $2.84, a 19% 

In [14]:
'"some"'.replace('"','').split(",")

['some']

In [17]:
chosen_route = ['a_agent','t_agent']
agent_results = {item: {item.replace("_agent","_result")} for item in chosen_route}
print(f"{agent_results}".replace("'",""))

{a_agent: {a_result}, t_agent: {t_result}}
